# Exercise 05: Classification Trees, Random Forest and Gradient Boosting – Heart Disease

**Dataset:** 918 patients from five hospitals (Cleveland, Hungary, Switzerland, Long Beach VA, Stalog), combined by
fedesoriano (2021) from the UCI heart disease data sets. Source:
[Kaggle – Heart Failure Prediction](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction).

| variable | meaning |
|---|---|
| `Age`, `Sex` | age (years), sex (M/F) |
| `ChestPainType` | TA typical angina, ATA atypical angina, NAP non-anginal pain, ASY asymptomatic |
| `RestingBP` | resting blood pressure (mm Hg) |
| `Cholesterol` | serum cholesterol (mg/dl) |
| `FastingBS` | fasting blood sugar > 120 mg/dl (1) or not (0) |
| `RestingECG` | resting ECG result (Normal, ST, LVH) |
| `MaxHR` | maximum heart rate achieved in the exercise test |
| `ExerciseAngina` | exercise-induced angina (Y/N) |
| `Oldpeak` | ST depression in the ECG under exercise |
| `ST_Slope` | slope of the ST segment under exercise (Up, Flat, Down) |
| `HeartDisease` | 1 = coronary heart disease diagnosed, 0 = normal – **target** |

**Business question:** Can routine examinations help to identify patients with heart disease? In a medical setting,
**missing a sick patient** (false negative) is much worse than a false alarm that leads to further tests.

**Slides:** Part 04 «Classification models» – classification trees (anatomy, Gini, growing, predicting), random forest &
gradient boosting for classification, evaluation; slide «Putting it together in scikit-learn».

**After this exercise you can ...**
- compute Gini impurity and the impurity decrease of a split by hand,
- fit, visualise, interpret and prune a classification tree,
- tune random forest and gradient boosting classifiers with `GridSearchCV`,
- compare classifiers with ROC/AUC and choose a threshold for a required recall.

## Concept

**Gini impurity** of a node with class shares $\hat p_k$:

$$G = 1 - \sum_{k=1}^{K} \hat p_k^2 \qquad (0 = \text{pure node},\; 0.5 = \text{maximally mixed for two classes})$$

**Impurity decrease** of a split into a left node $L$ and a right node $R$ (with $n_L + n_R = n$ observations):

$$\Delta G = G_{\text{parent}} - \Big(\frac{n_L}{n}\,G_L + \frac{n_R}{n}\,G_R\Big)$$

The tree chooses greedily the (feature, threshold) with the largest $\Delta G$ and repeats recursively until a stopping
rule applies (`max_depth`, `min_samples_leaf`, pure node). Entropy is an alternative impurity measure with almost the same
shape. Large trees can be pruned with `ccp_alpha`.

**Prediction.** A leaf predicts the class shares of its training observations, e.g. 145 sick : 125 healthy →
$\hat p(\text{sick}) = 145/270 \approx 0.54$ → predict "sick" (if $\tau = 0.5$). Trees output **probabilities**, which is
the basis for thresholds and ROC curves.

**Ensembles for classification.**
- *Random forest:* many deep trees on bootstrap samples with random feature subsets (default $m = \sqrt p$); the forest
  averages the class probabilities of the trees.
- *Gradient boosting:* many small trees, built one after the other; each tree fits the pseudo-residuals (negative
  gradient) of the log-loss.

## Libraries and settings

In [ ]:
# Libraries
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn import tree
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score, precision_score,
                             roc_auc_score, roc_curve, classification_report)
from sklearn.inspection import permutation_importance

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

## Exercise 1: Gini impurity by hand

A node contains 200 patients, 110 with heart disease and 90 without. Two candidate splits are evaluated:

| candidate split | left node (yes) | right node (no) |
|---|---|---|
| `ST_Slope = Flat?` | 100 patients: 80 sick / 20 healthy | 100 patients: 30 sick / 70 healthy |
| `ExerciseAngina = Y?` | 80 patients: 64 sick / 16 healthy | 120 patients: 46 sick / 74 healthy |

a) Compute the Gini impurity of the parent node.

b) Compute $G_L$, $G_R$, the weighted impurity and $\Delta G$ for both candidate splits. Which split does the tree choose?

c) What does the left leaf of the chosen split predict (class and probability)?

d) Verify your results with Python.

In [ ]:
def gini(n_sick, n_healthy):
    """Gini impurity of a node with two classes."""
    # Your code here
    pass

# Your code here

**Your answer:** ...

## Exercise 2: Import, check and prepare the data

a) Import the data and check the descriptive statistics. Which values of `RestingBP` and `Cholesterol` are
implausible?

b) Remove the row with `RestingBP == 0`. For `Cholesterol == 0` (presumably not measured) create an indicator
`Cholesterol_missing` and replace the zeros by the median of the measured values.

c) Create dummy variables for the categorical features (`pd.get_dummies(..., drop_first=True, dtype=int)`) and split the
data into a stratified training (80 %) and test set (20 %) with `random_state=42`.

d) What is the share of patients with heart disease?

In [ ]:
# a) Import the data
df = pd.read_csv('./Data/heart.csv', sep=',')
# Your code here

# b) Implausible values
# Your code here

# c) Dummy variables and train/test split
X = ___
y = df['HeartDisease']
X_train, X_test, y_train, y_test = ___

# d) Share of patients with heart disease
# Your code here

**Your answer:** ...

## Exercise 3: A small classification tree

a) Fit a `DecisionTreeClassifier(max_depth=3, random_state=42)` on the training set and visualise it with
`tree.plot_tree(..., class_names=['healthy', 'sick'], filled=True)`.

b) Interpret the root split and the second level. Do the splits make medical sense?

c) Check the root split by hand: the code below prints the class counts of the root and its two children. Compute the
Gini values and $\Delta G$ and compare them with the plot.

d) Compute accuracy, recall and AUC on the test set.

In [ ]:
# a) Fit and plot the tree
clf_tree = ___
# Your code here

# c) Class counts of the root and its children
t = clf_tree.tree_
for node, name in [(0, 'root'), (t.children_left[0], 'left child'), (t.children_right[0], 'right child')]:
    counts = np.round(t.value[node][0] * t.n_node_samples[node]).astype(int)
    print(f'{name:12s}: n = {t.n_node_samples[node]}, healthy = {counts[0]}, sick = {counts[1]}, Gini = {t.impurity[node]:.3f}')

# d) Test metrics
# Your code here

**Your answer:** ...

## Exercise 4: Overfitting and pruning

a) Fit a tree without depth limit. Compare training and test accuracy with the depth-3 tree. How many leaves does the
full tree have?

b) Choose `ccp_alpha` with `GridSearchCV` (stratified 5-fold CV, `scoring='roc_auc'`) from the values of
`cost_complexity_pruning_path(X_train, y_train).ccp_alphas`. How many leaves does the best tree have, and what is its test
AUC?

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# a) Full tree vs. depth-3 tree
# Your code here

# b) Pruning with GridSearchCV
# Your code here

**Your answer:** ...

## Exercise 5: Random forest and gradient boosting with GridSearchCV

Tune the two ensembles as on the slide «Putting it together in scikit-learn» (stratified 5-fold CV,
`scoring='roc_auc'`):

- `RandomForestClassifier(n_estimators=500, random_state=42)` with
  `{'max_features': ['sqrt', 0.5], 'min_samples_leaf': [1, 5, 20]}`
- `GradientBoostingClassifier(random_state=42)` with
  `{'learning_rate': [0.05, 0.1], 'n_estimators': [100, 300], 'max_depth': [2, 3]}`

Print the best parameters and the best CV AUC for both models. Which hyperparameters matter?

In [ ]:
# Random forest
grid_rf = GridSearchCV(___)
# Your code here

# Gradient boosting
grid_gb = GridSearchCV(___)
# Your code here

**Your answer:** ...

## Exercise 6: Model comparison on the test set

a) Add a logistic regression (with `StandardScaler`, in a `make_pipeline`) as a baseline. Plot the ROC curves of the
depth-3 tree, the pruned tree, the tuned random forest, the tuned gradient boosting and the logistic regression on the
test set, and print the AUCs.

b) For the best model: which threshold $\tau$ is needed to reach a **recall of at least 0.95**? What are the precision and
the false positive rate at this threshold? Compare with $\tau = 0.5$.

In [ ]:
models = {
    'Tree (depth 3)':      clf_tree,
    'Pruned tree':         pruned_tree,
    'Random forest':       grid_rf.best_estimator_,
    'Gradient boosting':   grid_gb.best_estimator_,
    'Logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(X_train, y_train),
}

# a) ROC curves and AUC
# Your code here

# b) Threshold for recall >= 0.95
# Your code here

**Your answer:** ...

## Exercise 7: What drives the predictions?

Compute the **permutation importance** of the tuned random forest on the test set (`scoring='roc_auc'`, 20 repeats) and
plot the 10 most important features. Compare with the splits of the depth-3 tree.

In [ ]:
# Your code here

**Your answer:** ...

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')